# 02 — Feature Engineering and Activation-Style Labels

**Project:** Agentic Campaign Strategist — Audience-to-Campaign Fit Advisor  
**Team:** Matt Hashemi, Marwah Faraj, Rebecca Cloe (AAI-590, Group 5)  
**Author of this notebook:** TODO  

**Purpose.** Convert cleaned Complete Journey transactions into weekly customer journey sequences, and generate activation-style labels using future-window weak supervision. This notebook produces the model-ready dataset used by notebooks 03 and 04.

**Inputs:** `data/processed/` output from notebook 01.  
**Outputs:** sequence arrays, labels, and a sample index saved to `data/processed/`.

## 1. Setup

In [ ]:
# Setup: make the src/ package importable.
# Run once from the repo root if not installed yet:  pip install -e .
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.append(str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 7
np.random.seed(RANDOM_STATE)

## 2. Load Cleaned Transactions

Load the cleaned dataset produced by notebook 01. We assert the expected schema so a wrong or synthetic input fails loudly instead of silently.

In [ ]:
from campaign_strategist.config import PROCESSED_DATA_DIR

transactions = pd.read_parquet(PROCESSED_DATA_DIR / 'transactions_clean.parquet')

expected_cols = {'household_id','week','product_category','quantity','sales_value','discount_value','used_coupon'}
assert expected_cols.issubset(transactions.columns), f'Missing columns: {expected_cols - set(transactions.columns)}'
print(f'{len(transactions):,} rows | {transactions.household_id.nunique():,} households | weeks {transactions.week.min()}-{transactions.week.max()}')

## 3. Build Weekly Journey Sequences

Each household-week becomes one time step with activity, spend, trips, discount rate, coupon rate, and category share features. Sliding windows of `SEQUENCE_LENGTH` weeks form one sample.

*Discussion (TODO): justify sequence length and horizon choices.*

In [ ]:
SEQUENCE_LENGTH = 12   # weeks of history per sample
PREDICTION_HORIZON = 4 # future weeks used ONLY for labeling

# TODO: call the (updated) feature builder from src/
# from campaign_strategist.features import build_feature_bundle
# bundle = build_feature_bundle(transactions, sequence_length=SEQUENCE_LENGTH, prediction_horizon=PREDICTION_HORIZON)

## 4. Label Design: Future-Window Weak Supervision

**Key fix vs. the earlier prototype:** labels must be computed **only from the future horizon window** (the weeks *after* each input sequence). The model inputs stay strictly in the past. This removes the label leakage where labels and inputs shared the same signals.

| Label | Future-window rule (draft) |
| --- | --- |
| new_customer_onboarding | household first became active recently (tenure at end of window <= 8 weeks) |
| win_back_reminder | active in input window, inactive in future window |
| price_led_coupon | high coupon/discount usage in future window |
| cross_sell_bundle | buys new categories in future window |
| loyalty_reward | consistently active and high spend in future window |
| seasonal_spotlight | activity concentrated in seasonal future weeks |

*TODO: finalize thresholds with the team and document each rule here.*

In [ ]:
# TODO: implement/import the future-window labeling function
# labels = label_from_future_window(...)

## 5. Label Distribution Analysis

Check class balance. The earlier prototype had 67% of samples in one class; if that happens again, tune rule thresholds and document the trade-offs.

In [ ]:
# label_counts = pd.Series(labels).value_counts()
# ax = label_counts.plot(kind='bar')
# ax.set_title('Activation-style label distribution')
# ax.set_ylabel('Samples')
# plt.tight_layout(); plt.show()

## 6. Save Model-Ready Outputs

In [ ]:
# np.save(PROCESSED_DATA_DIR / 'sequences_x.npy', x)
# np.save(PROCESSED_DATA_DIR / 'labels_y.npy', y)
# sample_index.to_parquet(PROCESSED_DATA_DIR / 'sample_index.parquet')

## 7. Summary and Next Steps

*TODO: summarize sample counts, label balance, and any concerns for modeling.*

## References

*TODO: cite 84.51°/dunnhumby Complete Journey, pandas, NumPy, and any methods referenced.*